# LVC Timeline FAQ RAG

Levi's Korea의 LVC Timeline 페이지를 직접 크롤링하여
페이지에 포함된 JEAN/JEANS 모델 정보를 수집하고,
수집한 원문을 기반으로 FAQ를 생성한다.

이후 FAQ와 크롤링 원문을 OpenAI Embedding으로 벡터화하여
Chroma DB에 저장하고, 사용자의 질문에 따라 관련 정보를 검색한 뒤
답변을 생성하는 RAG 시스템을 구현한다.

Source:
https://levi.co.kr/pages/lvc_timeline.html

## 1. import + 환경설정

In [1]:
!pip -q install requests beautifulsoup4 openai python-dotenv chromadb

In [ ]:
import os
import re
import json
import requests

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI

from pathlib import Path

env_candidates = [
    Path("../.env"),
    Path("../../.env"),
    Path(".env")
]

for env_path in env_candidates:
    if env_path.exists():
        load_dotenv(env_path)
        break

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY가 필요합니다."

client = OpenAI(api_key=OPENAI_API_KEY)

URL = "https://levi.co.kr/pages/lvc_timeline.html"

## 2. Web Crawling

In [34]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/128.0 Safari/537.36"
    )
}

response = requests.get(
    URL,
    headers=HEADERS,
    timeout=10
)

print("status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

status code: 200


## 3. LVC Text Search in the page

In [40]:
from bs4 import NavigableString, Tag

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()


all_product_headings = soup.find_all("h4")

print("전체 상품 제목 수:", len(all_product_headings))

for h in all_product_headings:
    print(clean_text(h.get_text(" ", strip=True)))

전체 상품 제목 수: 15
1890 XX 501 ® JEANS
1937 501 ® JEANS
1944 501 ® JEANS
1947 501 ® JEANS
1954 501 ® JEANS
1955 501 ® JEANS
1966 501 ® JEANS
1976 501 ® JEANS
1984 501 ® JEANS
1880S TRIPLE PLEAT BLOUSE
1936 TYPE I JACKET
1953 TYPE II JACKET MEN’S
1961 557 TRUCKER JACKET
1950’S 701 ® JEAN WOMEN’S
아이디 찾기


In [45]:
models = []

for h in all_product_headings:
    heading = clean_text(h.get_text(" ", strip=True))

    # JEAN 또는 JEANS가 들어간 바지 상품만 선택
    if not re.search(r"\bJEANS?\b", heading, re.IGNORECASE):
        continue

    content_parts = [heading]

    # 현재 상품 제목 다음부터 다음 h4 상품 제목 전까지 수집
    for elem in h.next_elements:

        if isinstance(elem, Tag):
            # 다음 상품 제목이 나오면 현재 모델 수집 종료
            if elem.name == "h4" and elem is not h:
                break

        elif isinstance(elem, NavigableString):
            if h in elem.parents:
                continue

            text = clean_text(str(elem))

            if not text:
                continue

            # 페이지 footer 시작 시 수집 종료
            if text in {
                "TRUCKER",
                "WOMEN’S",
                "WOMEN'S",
                "시즈널 컬렉션",
                "리바이스 아카이브",
                "ABOUT LVC",
                "아이디 찾기"
            }:
                break

            content_parts.append(text)

    content = clean_text(" ".join(content_parts))

    year_match = re.search(r"\d{4}", heading)
    year = year_match.group() if year_match else "unknown"

    models.append({
        "year": year,
        "model": heading,
        "content": content,
        "source_url": URL
    })


print("수집된 바지 모델 수:", len(models))

for model in models:
    print("\n" + "=" * 80)
    print("YEAR :", model["year"])
    print("MODEL:", model["model"])
    print(model["content"][:700])

수집된 바지 모델 수: 10

YEAR : 1890
MODEL: 1890 XX 501 ® JEANS
1890 XX 501 ® JEANS SIZE 허리 28/30/32/34/36 inch 바지길이 32/34 inch 밑위 13 inch 무릎 폭 18 inch 밑단 폭 18 inch *xx는 상품에 따라 다를 수 있습니다. SHOP NOW DETAIL 1890 XX 501®는 현재 단종된 상품이지만, LVC의 역사적인 모델을 이야기할 때 빼놓을 수 없는 중요한 아이템입니다. 1890년은 501®이라는 LOT 번호가 처음 도입된 해입니다. 당시 리바이스 청바지에 적용되던 리벳 특허가 같은 해 만료되면서, 다른 회사들도 리벳을 사용할 수 있게 되었습니다. 이에 리바이스는 상품에 번호를 부여해 품질을 강조하고자 했습니다. 숫자 '5'는 최고급 라인을 의미하며, 안주머니 겉면에는 **XX(Extra Strong, Double Extra Heavy를 뜻하는 표기)**의 독창성을 상징하는 마크가 새겨졌습니다. 1890 XX 501 ® 의 핏은 하이 웨이스트의 와이드한 레그가 특징 9oz 플레인 셀비지 데님 원단 사용 하나의 백포켓과 돌출 리벳 사용 신치백(뒷고리)를 사용하여 허리 사이즈에 맞춰 늘렸다 줄였다 가능 서스펜더 버튼 부착 최초의 TWO HORSE PULL 가죽패치 부착 크로치 리벳(지퍼 아래쪽 터짐을 방지하기 위한 리벳 추가) 셀비지 쪽에 레드 스티치가 없으며 밑단 마무리의 경우 체인스티치가 아닌 싱글스티치로 되어 있음 포켓 뒷면에 상

YEAR : 1937
MODEL: 1937 501 ® JEANS
1937 501 ® JEANS SIZE 허리 28/30/32/34/36 inch 바지길이 32/34 inch 밑위 13 inch 무릎 폭 19 inch 밑단 폭 18 inch *xx는 상품에 따라 다를 수 있습니다. SHOP NOW DETAIL 1937년 미국의 사회적 이슈였던 대공황을 벗어나면서, 리바이스 501 ® 청바지는 낡은 것과 새로운 것이 공존한 청바지로 

In [46]:
os.makedirs("source_data", exist_ok=True)

with open(
    "source_data/lvc_models.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(models, f, ensure_ascii=False, indent=2)

print("저장 완료: source_data/lvc_models.json")

저장 완료: source_data/lvc_models.json


## 4. FAQ Generation

크롤링한 LVC 바지 모델별 원문을 바탕으로
각 모델에 대한 FAQ를 생성한다.

FAQ 생성 시 외부 지식은 사용하지 않고,
크롤링한 원문에 포함된 정보만 사용하도록 제한한다.

In [48]:
def generate_faqs(model_data, num_faqs=4):
    prompt = f"""
아래는 Levi's Korea 공식 LVC Timeline 페이지에서 크롤링한
{model_data['model']}의 원문입니다.

[원문]
{model_data['content']}

위 원문만 사용하여 FAQ {num_faqs}개를 생성하세요.

규칙:
1. 외부 지식을 사용하지 마세요.
2. 답변은 반드시 원문에서 확인할 수 있어야 합니다.
3. 모델의 핏, 원단, 역사적 배경, 디테일 등을 골고루 질문하세요.
4. 질문과 답변은 한국어로 작성하세요.
5. 다음 JSON 형식으로만 반환하세요.

{{
  "faqs": [
    {{
      "question": "질문",
      "answer": "답변"
    }}
  ]
}}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": "주어진 원문만 근거로 FAQ를 생성하는 도우미입니다."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return json.loads(response.choices[0].message.content)

## 5. 전체 바지 모델 FAQ 생성

In [49]:
all_faqs = []

for model_data in models:
    result = generate_faqs(model_data, num_faqs=4)

    for faq in result["faqs"]:
        all_faqs.append({
            "year": model_data["year"],
            "model": model_data["model"],
            "question": faq["question"],
            "answer": faq["answer"],
            "source_url": model_data["source_url"]
        })

    print(
        f"{model_data['model']} FAQ 생성 완료 "
        f"({len(result['faqs'])}개)"
    )

print("\n총 FAQ 수:", len(all_faqs))

1890 XX 501 ® JEANS FAQ 생성 완료 (4개)
1937 501 ® JEANS FAQ 생성 완료 (4개)
1944 501 ® JEANS FAQ 생성 완료 (4개)
1947 501 ® JEANS FAQ 생성 완료 (4개)
1954 501 ® JEANS FAQ 생성 완료 (4개)
1955 501 ® JEANS FAQ 생성 완료 (4개)
1966 501 ® JEANS FAQ 생성 완료 (4개)
1976 501 ® JEANS FAQ 생성 완료 (4개)
1984 501 ® JEANS FAQ 생성 완료 (4개)
1950’S 701 ® JEAN WOMEN’S FAQ 생성 완료 (4개)

총 FAQ 수: 40


## 6. FAQ 내용 직접 확인

In [50]:
for faq in all_faqs:
    print(f"\n[{faq['model']}]")
    print("Q:", faq["question"])
    print("A:", faq["answer"])


[1890 XX 501 ® JEANS]
Q: 1890 XX 501®의 핏은 어떤 특징이 있나요?
A: 1890 XX 501®의 핏은 하이 웨이스트의 와이드한 레그가 특징입니다.

[1890 XX 501 ® JEANS]
Q: 1890 XX 501®는 어떤 원단으로 만들어졌나요?
A: 1890 XX 501®는 9oz 플레인 셀비지 데님 원단을 사용합니다.

[1890 XX 501 ® JEANS]
Q: 1890 XX 501®의 역사적 배경은 무엇인가요?
A: 1890년은 501®이라는 LOT 번호가 처음 도입된 해이며, 당시 리바이스 청바지에 적용되던 리벳 특허가 같은 해 만료되어 다른 회사들도 리벳을 사용할 수 있게 되었습니다.

[1890 XX 501 ® JEANS]
Q: 1890 XX 501®의 디테일에는 어떤 것들이 있나요?
A: 1890 XX 501®는 하나의 백포켓과 돌출 리벳, 신치백(뒷고리), 서스펜더 버튼, 최초의 TWO HORSE PULL 가죽패치, 크로치 리벳, 셀비지 쪽에 레드 스티치가 없으며, 밑단 마무리는 싱글스티치로 되어 있습니다.

[1937 501 ® JEANS]
Q: 1937 501 ® JEANS의 핏은 어떤 특징이 있나요?
A: 1937년 501 ® 의 핏은 둥근 윗단의 스트레이트 레그가 특징입니다.

[1937 501 ® JEANS]
Q: 1937 501 ® JEANS는 어떤 원단으로 만들어졌나요?
A: 1937 501 ® JEANS는 10oz 레드 셀비지 데님원단을 사용하였습니다.

[1937 501 ® JEANS]
Q: 1937년 501 ® JEANS의 역사적 배경은 무엇인가요?
A: 1937년 미국의 사회적 이슈였던 대공황을 벗어나면서, 리바이스 501 ® 청바지는 낡은 것과 새로운 것이 공존한 청바지로 진화하였습니다.

[1937 501 ® JEANS]
Q: 1937 501 ® JEANS의 주요 디테일은 무엇인가요?
A: 1937 501 ® 에는 두 개의 백포켓과 히든 리벳이 추가되었으며, 서스펜더의 사라짐에 따라 벨트 

## 7. FAQ 검수 및 수정

생성된 FAQ를 원문과 비교하여 확인하고,
모델 비교에 더 유용하도록 일부 질문과 답변을 수정한다.

In [51]:
# 생성된 FAQ 중 일부 답변/질문 보정
for faq in all_faqs:

    # 1955 원단 정보 구체화
    if faq["year"] == "1955" and "원단" in faq["question"]:
        faq["answer"] = (
            "1955 501은 12oz 레드 셀비지 데님을 사용합니다."
        )

    # 1950's 701의 모호한 질문 표현 수정
    if faq["year"] == "1950" and faq["question"].startswith("이 청바지의 원단"):
        faq["question"] = "1950’S 701 ® JEAN WOMEN’S의 원단은 무엇인가요?"

    if faq["year"] == "1950" and faq["question"].startswith("이 청바지의 디테일"):
        faq["question"] = "1950’S 701 ® JEAN WOMEN’S의 주요 디테일은 무엇인가요?"


# RAG 검색에 특히 유용한 핵심 FAQ 추가
extra_faqs = [
    {
        "year": "1937",
        "model": "1937 501 ® JEANS",
        "question": "레드탭이 처음 등장한 501 모델은 무엇인가요?",
        "answer": (
            "1937 501입니다. Levi's 청바지를 경쟁사 제품과 구별하기 위해 "
            "오른쪽 뒷주머니에 처음으로 레드탭을 부착하기 시작했습니다."
        ),
        "source_url": URL
    },

    {
        "year": "1944",
        "model": "1944 501 ® JEANS",
        "question": "1944 501은 왜 아치형 스티치를 프린트로 만들었나요?",
        "answer": (
            "제2차 세계대전 중 자재 절약을 위해 기존 아치형 스티치를 제거하고 "
            "뒷주머니에 아치형 디자인을 프린트했습니다."
        ),
        "source_url": URL
    },

    {
        "year": "1954",
        "model": "1954 501 ® JEANS",
        "question": "지퍼 플라이가 처음 적용된 501 모델은 무엇인가요?",
        "answer": (
            "1954 501입니다. 동부 해안 소비자에게 익숙하지 않았던 버튼 플라이 대신 "
            "지퍼를 적용한 501 Z를 출시했습니다."
        ),
        "source_url": URL
    },

    {
        "year": "1966",
        "model": "1966 501 ® JEANS",
        "question": "백포켓 리벳을 바텍 스티치로 대체한 501 모델은 무엇인가요?",
        "answer": (
            "1966 501입니다. 백포켓의 리벳을 바텍 스티치로 대체하여 "
            "내구성을 유지하면서 가구 손상 문제를 줄였습니다."
        ),
        "source_url": URL
    },

    {
        "year": "1976",
        "model": "1976 501 ® JEANS",
        "question": "1976 501의 레드탭은 이전 모델과 어떻게 달라졌나요?",
        "answer": (
            "1971년부터 레드탭의 로고가 대문자 E의 LEVI'S에서 "
            "소문자 e가 포함된 Levi's로 변경되었습니다."
        ),
        "source_url": URL
    },

    {
        "year": "1984",
        "model": "1984 501 ® JEANS",
        "question": "1984 501은 이전 셀비지 원단과 어떤 변화가 있었나요?",
        "answer": (
            "기술 발전으로 더 폭이 넓은 원단을 사용할 수 있게 되었고, "
            "Shrink-to-Fit 501에도 이러한 넓은 원단이 사용되었습니다."
        ),
        "source_url": URL
    },

    {
        "year": "1950",
        "model": "1950’S 701 ® JEAN WOMEN’S",
        "question": "1950’S 701에는 어떤 지퍼가 사용되나요?",
        "answer": (
            "1950’S 701에는 1893년부터 제조되고 있는 Talon 지퍼가 사용됩니다."
        ),
        "source_url": URL
    }
]


# 같은 셀을 여러 번 실행해도 FAQ가 중복되지 않도록 처리
combined_faqs = all_faqs + extra_faqs

unique_faqs = {}

for faq in combined_faqs:
    key = (
        faq["year"],
        faq["question"].strip()
    )
    unique_faqs[key] = faq

all_faqs = list(unique_faqs.values())

print("FAQ 검수 및 보강 완료")
print("최종 FAQ 수:", len(all_faqs))

FAQ 검수 및 보강 완료
최종 FAQ 수: 47


## 8. FAQ 저장

In [53]:
with open(
    "source_data/lvc_faqs.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        all_faqs,
        f,
        ensure_ascii=False,
        indent=2
    )

print("저장 완료: source_data/lvc_faqs.json")
print("저장된 FAQ 수:", len(all_faqs))

저장 완료: source_data/lvc_faqs.json
저장된 FAQ 수: 47


In [3]:
with open("source_data/lvc_models.json", "r", encoding="utf-8") as f:
    models = json.load(f)

with open("source_data/lvc_faqs.json", "r", encoding="utf-8") as f:
    all_faqs = json.load(f)

print("모델 수:", len(models))
print("FAQ 수:", len(all_faqs))

모델 수: 10
FAQ 수: 47


## 9. Embedding & Chroma DB

생성한 FAQ의 질문·답변과 크롤링한 공식 원문 Chunk를
OpenAI Embedding 모델을 이용해 벡터로 변환한다.

FAQ와 원문 Chunk를 함께 Chroma DB에 저장하여,
FAQ에 직접 포함되지 않은 세부 정보도 검색할 수 있도록 구성한다.

In [4]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="chroma_db"
)


def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []

    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


print("Embedding / Chroma 준비 완료")

Embedding / Chroma 준비 완료


In [56]:
try:
    chroma_client.delete_collection(name="lvc_faqs")
except:
    pass

collection = chroma_client.get_or_create_collection(
    name="lvc_faqs"
)

print("Chroma DB 초기화 완료")

Chroma DB 초기화 완료


In [57]:
for i, faq in enumerate(all_faqs):

    document = (
        f"모델: {faq['model']}\n"
        f"질문: {faq['question']}\n"
        f"답변: {faq['answer']}"
    )

    embedding = get_embedding(document)

    collection.upsert(
        ids=[f"faq_{i}"],
        documents=[document],
        embeddings=[embedding],
        metadatas=[{
            "doc_type": "faq",
            "year": faq["year"],
            "model": faq["model"],
            "source_url": faq["source_url"]
        }]
    )

print("FAQ 저장 완료:", len(all_faqs))

FAQ 저장 완료: 47


In [58]:
source_count = 0

for model in models:
    chunks = chunk_text(model["content"])

    for j, chunk in enumerate(chunks):

        document = (
            f"모델: {model['model']}\n"
            f"공식 원문: {chunk}"
        )

        embedding = get_embedding(document)

        collection.upsert(
            ids=[f"source_{model['year']}_{j}"],
            documents=[document],
            embeddings=[embedding],
            metadatas=[{
                "doc_type": "source",
                "year": model["year"],
                "model": model["model"],
                "source_url": model["source_url"]
            }]
        )

        source_count += 1


print("원문 Chunk 저장 완료:", source_count)
print("Chroma DB 전체 문서 수:", collection.count())

원문 Chunk 저장 완료: 23
Chroma DB 전체 문서 수: 70


## 10. Retrieval

질문의 형태에 따라 검색 방식을 다르게 적용한다.

- 특정 연도가 포함된 질문은 해당 모델의 공식 원문을 검색한다.
- 전체 모델 비교나 목록 질문은 전체 공식 원문을 확인한다.
- 그 외의 질문은 Chroma DB에서 의미 기반 벡터 검색을 수행한다.

In [15]:
def retrieve_documents(query, n_results=8):
    query_embedding = get_embedding(query)

    available_years = sorted({
        model["year"]
        for model in models
        if model["year"] != "unknown"
    })

    year_pattern = "(" + "|".join(
        map(re.escape, available_years)
    ) + ")"

    years = re.findall(year_pattern, query)
    years = list(dict.fromkeys(years))

    # 1. 질문에 특정 연도가 있는 경우
    # → 해당 연도의 공식 원문 전체 사용
    if years:
        all_documents = []
        all_metadatas = []

        for year in years:
            results = collection.get(
                where={
                    "$and": [
                        {"year": {"$eq": year}},
                        {"doc_type": {"$eq": "source"}}
                    ]
                },
                include=["documents", "metadatas"]
            )

            all_documents.extend(results["documents"])
            all_metadatas.extend(results["metadatas"])

        return {
            "documents": [all_documents],
            "metadatas": [all_metadatas]
        }

    # 2. 전체 모델을 확인해야 하는 질문인지 판단
    global_keywords = [
        "가장",
        "모델들",
        "모델들은",
        "어떤 것들",
        "어떤 모델들",
        "모두",
        "전부",
        "전체"
    ]

    is_global_query = any(
        keyword in query
        for keyword in global_keywords
    )

    # 전체 비교/목록 질문
    # → 10개 모델의 공식 원문 Chunk를 전부 사용
    if is_global_query:
        results = collection.get(
            where={"doc_type": {"$eq": "source"}},
            include=["documents", "metadatas"]
        )

        return {
            "documents": [results["documents"]],
            "metadatas": [results["metadatas"]]
        }

    # 3. 일반적인 연도 없는 질문
    # → 의미 기반 벡터 검색
    return collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

In [9]:
query = "지퍼가 처음 들어가기 시작한 501 모델은 뭐야?"

results = retrieve_documents(query)

print("질문:", query)

for i in range(len(results["documents"][0])):
    print("\n--- 검색 결과", i + 1, "---")
    print(results["documents"][0][i])
    print("metadata:", results["metadatas"][0][i])

질문: 지퍼가 처음 들어가기 시작한 501 모델은 뭐야?

--- 검색 결과 1 ---
모델: 1954 501 ® JEANS
질문: 지퍼 플라이가 처음 적용된 501 모델은 무엇인가요?
답변: 1954 501입니다. 동부 해안 소비자에게 익숙하지 않았던 버튼 플라이 대신 지퍼를 적용한 501 Z를 출시했습니다.
metadata: {'year': '1954', 'doc_type': 'faq', 'source_url': 'https://levi.co.kr/pages/lvc_timeline.html', 'model': '1954 501 ® JEANS'}

--- 검색 결과 2 ---
모델: 1937 501 ® JEANS
질문: 레드탭이 처음 등장한 501 모델은 무엇인가요?
답변: 1937 501입니다. Levi's 청바지를 경쟁사 제품과 구별하기 위해 오른쪽 뒷주머니에 처음으로 레드탭을 부착하기 시작했습니다.
metadata: {'year': '1937', 'source_url': 'https://levi.co.kr/pages/lvc_timeline.html', 'doc_type': 'faq', 'model': '1937 501 ® JEANS'}

--- 검색 결과 3 ---
모델: 1890 XX 501 ® JEANS
질문: 1890 XX 501®의 역사적 배경은 무엇인가요?
답변: 1890년은 501®이라는 LOT 번호가 처음 도입된 해이며, 당시 리바이스 청바지에 적용되던 리벳 특허가 같은 해 만료되어 다른 회사들도 리벳을 사용할 수 있게 되었습니다.
metadata: {'source_url': 'https://levi.co.kr/pages/lvc_timeline.html', 'model': '1890 XX 501 ® JEANS', 'doc_type': 'faq', 'year': '1890'}

--- 검색 결과 4 ---
모델: 1955 501 ® JEANS
질문: 1955 501 ® JEANS의 역사적 배경은 무엇인가요?
답변: 당시 젊은이들은 501 ® 

## 11. RAG Answer Generation

검색된 공식 원문 및 FAQ와 함께,
크롤링 원문에서 검수한 모델별 구조화 정보를 이용하여 최종 답변을 생성한다.

밑위, 무릎 폭, 밑단 폭, 원단 중량, 플라이, 레드탭과 같이
정확한 비교가 필요한 정보는 구조화된 데이터를 이용하여 Python 코드로 직접 처리한다.

반면 역사적 배경, 디자인 변화의 이유, 모델 간 특징 비교와 같은
설명형 질문은 Chroma DB에서 검색한 Context를 기반으로 GPT가 답변하도록 구성한다.

이를 통해 수치 및 조건 검색의 정확성을 높이면서,
자연어 기반 RAG의 설명 능력도 함께 활용한다.

In [41]:
# ============================================================
# 검수된 모델별 구조화 정보
# 크롤링한 LVC Timeline 원문을 기준으로 정리
# ============================================================

model_specs = {
    "1890 XX 501 ® JEANS": {
        "rise": 13,
        "knee": 18,
        "hem": 18,
        "fabric_oz": 9,
        "fly": None,
        "red_tab": "없음"
    },

    "1937 501 ® JEANS": {
        "rise": 13,
        "knee": 19,
        "hem": 18,
        "fabric_oz": 10,
        "fly": "버튼 플라이",
        "red_tab": "대문자 E 단면"
    },

    "1944 501 ® JEANS": {
        "rise": 13,
        "knee": 17,
        "hem": 17,
        "fabric_oz": 12,
        "fly": "버튼 플라이",
        "red_tab": "대문자 E 단면"
    },

    "1947 501 ® JEANS": {
        "rise": 12,
        "knee": 18,
        "hem": 16,
        "fabric_oz": 12,
        "fly": "버튼 플라이",
        "red_tab": "대문자 E 단면"
    },

    "1954 501 ® JEANS": {
        "rise": 12,
        "knee": 18,
        "hem": 15,
        "fabric_oz": 12,
        "fly": "지퍼 플라이 (501 Z)",
        "red_tab": "대문자 E 단면"
    },

    "1955 501 ® JEANS": {
        "rise": 13,
        "knee": 20,
        "hem": 17,
        "fabric_oz": 12,
        "fly": "버튼 플라이",
        "red_tab": "대문자 E 양면"
    },

    "1966 501 ® JEANS": {
        "rise": 12,
        "knee": 19,
        "hem": 17,
        "fabric_oz": 12,
        "fly": "버튼 플라이",
        "red_tab": "대문자 E 양면"
    },

    "1976 501 ® JEANS": {
        "rise": 12,
        "knee": 19,
        "hem": 17,
        "fabric_oz": 12,
        "fly": "버튼 플라이",
        "red_tab": "소문자 e 양면"
    },

    "1984 501 ® JEANS": {
        "rise": 12,
        "knee": 18,
        "hem": 16,
        "fabric_oz": 12,
        "fly": "버튼 플라이",
        "red_tab": "소문자 e 양면"
    },

    "1950’S 701 ® JEAN WOMEN’S": {
        "rise": 13,
        "knee": 18,
        "hem": 17,
        "fabric_oz": 10,
        "fly": "지퍼 플라이 (Talon)",
        "red_tab": "대문자 E"
    }
}


# ============================================================
# 구조화 정보 표시
# ============================================================

def build_spec_table():
    lines = []

    for model_name, spec in model_specs.items():

        fabric = (
            f"{spec['fabric_oz']}oz"
            if spec["fabric_oz"] is not None
            else "원문에서 구조화하지 않음"
        )

        lines.append(
            f"- {model_name} | "
            f"밑위 {spec['rise']} inch | "
            f"무릎 {spec['knee']} inch | "
            f"밑단 {spec['hem']} inch | "
            f"원단 {fabric} | "
            f"플라이 {spec['fly']} | "
            f"레드탭 {spec['red_tab']}"
        )

    return "\n".join(lines)


# ============================================================
# 질문에서 수치 조건 추출
# ============================================================

def extract_numeric_filters(query):
    filters = {}

    rise_match = re.search(
        r"밑위(?:가|는|은|이)?\s*(\d+(?:\.\d+)?)\s*(?:인치|inch)",
        query,
        re.IGNORECASE
    )

    knee_match = re.search(
        r"무릎(?:\s*폭)?(?:가|는|은|이)?\s*(\d+(?:\.\d+)?)\s*(?:인치|inch)",
        query,
        re.IGNORECASE
    )

    hem_match = re.search(
        r"밑단(?:\s*폭)?(?:가|는|은|이)?\s*(\d+(?:\.\d+)?)\s*(?:인치|inch)",
        query,
        re.IGNORECASE
    )

    oz_match = re.search(
        r"(\d+(?:\.\d+)?)\s*oz",
        query,
        re.IGNORECASE
    )

    if rise_match:
        filters["rise"] = float(rise_match.group(1))

    if knee_match:
        filters["knee"] = float(knee_match.group(1))

    if hem_match:
        filters["hem"] = float(hem_match.group(1))

    if oz_match:
        filters["fabric_oz"] = float(oz_match.group(1))

    return filters


# ============================================================
# 정확한 조건 검색
# ============================================================

def find_models_by_filters(filters):
    matched = []

    for model_name, spec in model_specs.items():

        is_match = True

        for key, value in filters.items():

            spec_value = spec.get(key)

            if spec_value is None:
                is_match = False
                break

            if float(spec_value) != float(value):
                is_match = False
                break

        if is_match:
            matched.append(model_name)

    return matched


# ============================================================
# 최대 / 최소 검색
# ============================================================

def find_extreme_models(field, mode):
    valid_models = {
        model_name: spec[field]
        for model_name, spec in model_specs.items()
        if spec.get(field) is not None
    }

    if not valid_models:
        return [], None

    if mode == "max":
        target_value = max(valid_models.values())
    else:
        target_value = min(valid_models.values())

    matched = [
        model_name
        for model_name, value in valid_models.items()
        if value == target_value
    ]

    return matched, target_value


# ============================================================
# 정확성이 필요한 질문은 Python으로 먼저 처리
# ============================================================

def structured_answer(query):

    # --------------------------------------------------------
    # 1. 정확한 수치 조건 검색
    # --------------------------------------------------------

    filters = extract_numeric_filters(query)

    if filters:

        matched = find_models_by_filters(filters)

        list_keywords = [
            "전부",
            "모두",
            "찾아",
            "어떤 모델",
            "모델들"
        ]

        if any(keyword in query for keyword in list_keywords):

            if not matched:
                return "해당 조건을 모두 만족하는 모델은 없습니다."

            filter_names = {
                "rise": "밑위",
                "knee": "무릎 폭",
                "hem": "밑단 폭",
                "fabric_oz": "원단 중량"
            }

            conditions = []

            for key, value in filters.items():

                if key == "fabric_oz":
                    conditions.append(
                        f"{filter_names[key]} {value:g}oz"
                    )

                else:
                    conditions.append(
                        f"{filter_names[key]} {value:g} inch"
                    )

            condition_text = ", ".join(conditions)

            models_text = "\n".join(
                f"- {model}"
                for model in matched
            )

            return (
                f"{condition_text} 조건을 모두 만족하는 모델은 "
                f"다음과 같습니다:\n\n{models_text}"
            )


    # --------------------------------------------------------
    # 2. 가장 넓은 / 좁은 / 높은 / 낮은 / 최대 / 최소
    # --------------------------------------------------------

    field = None
    field_name = None

    if "무릎" in query:
        field = "knee"
        field_name = "무릎 폭"

    elif "밑단" in query:
        field = "hem"
        field_name = "밑단 폭"

    elif "밑위" in query:
        field = "rise"
        field_name = "밑위"

    if field is not None and (
        "가장" in query
        or "최대" in query
        or "최소" in query
    ):

        if (
            "좁" in query
            or "낮" in query
            or "작" in query
            or "최소" in query
        ):
            mode = "min"

        else:
            mode = "max"

        matched, value = find_extreme_models(
            field,
            mode
        )

        models_text = ", ".join(matched)

        return (
            f"{field_name} 기준으로 조건에 해당하는 모델은 "
            f"{models_text}이며, 값은 {value:g} inch입니다."
        )


    # --------------------------------------------------------
    # 3. 레드탭 관련 질문
    # --------------------------------------------------------

    if "레드탭" in query:

        # 레드탭 변화 흐름
        if (
            "흐름" in query
            or "연도순" in query
            or "변화" in query
            or "바뀌" in query
        ):

            return (
                "501 계열의 레드탭 변화는 다음과 같습니다.\n\n"
                "- 1937, 1944, 1947, 1954 501: 대문자 E 단면 레드탭\n"
                "- 1955, 1966 501: 대문자 E 양면 레드탭\n"
                "- 1976, 1984 501: 소문자 e 양면 레드탭\n\n"
                "1950’S 701 WOMEN’S는 별도의 여성용 모델이며 "
                "대문자 E 레드탭을 사용합니다."
            )

        # 레드탭 모델 목록
        list_keywords = [
            "모델들",
            "어떤 것들",
            "어떤 모델",
            "전부",
            "모두",
            "사용하는 모델"
        ]

        if any(keyword in query for keyword in list_keywords):

            matched = []

            for model_name, spec in model_specs.items():

                red_tab = spec["red_tab"]

                if "대문자 E" in query:
                    if "대문자 E" not in red_tab:
                        continue

                if "소문자 e" in query:
                    if "소문자 e" not in red_tab:
                        continue

                if "단면" in query:
                    if "단면" not in red_tab:
                        continue

                if "양면" in query:
                    if "양면" not in red_tab:
                        continue

                matched.append(model_name)

            if matched:

                models_text = "\n".join(
                    f"- {model}"
                    for model in matched
                )

                return (
                    "조건에 해당하는 레드탭 모델은 "
                    f"다음과 같습니다:\n\n{models_text}"
                )


    # --------------------------------------------------------
    # 4. 지퍼 관련 구조화 검색
    # --------------------------------------------------------

    if "지퍼" in query:

        matched = [
            model_name
            for model_name, spec in model_specs.items()
            if spec["fly"] is not None
            and "지퍼" in spec["fly"]
        ]

        # Talon 조건
        if "talon" in query.lower():

            matched = [
                model_name
                for model_name in matched
                if "talon" in model_specs[model_name]["fly"].lower()
            ]

        # 501 Z 조건
        if "501 z" in query.lower():

            matched = [
                model_name
                for model_name in matched
                if "501 z" in model_specs[model_name]["fly"].lower()
            ]

        # 여성용 조건
        if (
            "여성용" in query
            or "여성" in query
            or "women" in query.lower()
        ):

            matched = [
                model_name
                for model_name in matched
                if "WOMEN" in model_name.upper()
            ]

        # 여러 모델 / 특정 모델 검색 질문
        if "모델" in query:

            if not matched:
                return "해당 조건을 만족하는 지퍼 모델은 없습니다."

            if len(matched) == 1:

                model = matched[0]

                return (
                    f"해당 조건을 만족하는 모델은 "
                    f"{model}입니다. "
                    f"플라이 정보는 {model_specs[model]['fly']}입니다."
                )

            models_text = "\n".join(
                f"- {model}: {model_specs[model]['fly']}"
                for model in matched
            )

            return (
                "지퍼 플라이를 사용하는 모델은 "
                f"다음과 같습니다:\n\n{models_text}"
            )


    # 구조화 검색으로 직접 처리할 필요 없는 질문
    return None


# ============================================================
# 최종 RAG Answer
# ============================================================

def rag_answer(query, n_results=8):

    # 1. Retrieval
    results = retrieve_documents(
        query,
        n_results=n_results
    )

    contexts = results["documents"][0]
    metadatas = results["metadatas"][0]


    # 2. 정확한 수치 / 목록 / 조건 질문 우선 처리
    exact_answer = structured_answer(query)

    if exact_answer is not None:

        return {
            "answer": exact_answer,
            "contexts": contexts,
            "metadatas": metadatas
        }


    # 3. 검색 결과를 모델별로 묶기
    grouped_contexts = {}

    for context, metadata in zip(contexts, metadatas):

        model_name = metadata["model"]

        if model_name not in grouped_contexts:
            grouped_contexts[model_name] = []

        if context not in grouped_contexts[model_name]:
            grouped_contexts[model_name].append(context)


    # 4. 크롤링된 모델 순서대로 Context 구성
    model_order = [
        model["model"]
        for model in models
    ]

    context_parts = []

    for model_name in model_order:

        if model_name not in grouped_contexts:
            continue

        model_context = "\n".join(
            grouped_contexts[model_name]
        )

        context_parts.append(
            f"""
==================================================
[MODEL: {model_name}]
==================================================
{model_context}
"""
        )

    context_text = "\n".join(context_parts)


    # 5. 구조화 스펙
    spec_table = build_spec_table()


    # 6. 최종 프롬프트
    prompt = f"""
아래 정보는 Levi's Korea 공식 LVC Timeline 페이지에서
직접 크롤링한 원문과 해당 원문을 기반으로 생성한 FAQ입니다.

또한 반복적으로 사용되는 모델별 수치와 디테일은
크롤링 원문을 검수하여 [구조화 스펙]으로 정리했습니다.

규칙:
1. 반드시 제공된 Context와 구조화 스펙만 사용하세요.

2. 구조화 스펙과 일반 Context가 동시에 존재하는 경우,
   수치와 명시적 제품 사양은 구조화 스펙을 우선하세요.

3. 질문에 여러 모델이 등장하면
   각 모델의 정보를 각각 확인하세요.

4. 수치 비교 질문에서는
   관련된 각 모델의 값을 먼저 확인한 뒤 답하세요.

5. 모델의 역사적 배경, 변경 이유, 디자인 변화와 같은
   설명형 질문은 Context를 근거로 답하세요.

6. 과거 특징과 현재 모델의 특징을 혼동하지 마세요.

7. 제공된 정보에 없는 내용은 추측하지 마세요.

8. 정말 확인할 수 없는 경우에만
   "제공된 정보만으로는 확인할 수 없습니다."라고 답하세요.

9. 답변은 한국어로 간결하고 자연스럽게 작성하세요.


[구조화 스펙]

{spec_table}


[검색된 Context]

{context_text}


[질문]

{query}
"""

    # 7. GPT 답변 생성
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "검색된 공식 원문과 검수된 구조화 스펙만 "
                    "근거로 정확하게 답변하세요. "
                    "숫자나 제품 사양은 구조화 스펙을 우선하세요."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )


    # 8. 결과 반환
    return {
        "answer": response.choices[0].message.content,
        "contexts": contexts,
        "metadatas": metadatas
    }

## 12. RAG Test

서로 다른 유형의 질문을 입력하여
RAG 시스템이 적절한 LVC 정보를 검색하고 답변하는지 확인한다.

In [37]:
test_questions = [
    # 역사적 배경 / 이유
    "1944 501은 왜 아치형 스티치를 프린트로 만들었어?",

    # 모델 간 특징 비교
    "1954 501과 1950’S 701은 지퍼와 관련해서 각각 어떤 특징이 있어?",

    # 수치 비교
    "1937과 1955 501의 데님 중량은 어떻게 달라?",

    # 구조화 조건 검색
    "대문자 E 양면 레드탭을 사용하는 모델은 어떤 것들이야?",

    # 최대값 검색
    "무릎 폭이 가장 넓은 모델은 뭐야?",

    # 핏 비교
    "1890 XX 501과 1955 501은 핏이 어떻게 달라?",

    # 여러 특징을 이용한 역검색
    "신치백은 없고 백포켓 리벳 대신 바텍을 사용하는 모델은 뭐야?",

    # 데이터에 없는 정보
    "1984 501의 세탁 후 평균 수축률은 몇 퍼센트야?",

    # 역사적 특징 기반 검색
    "501에 레드탭이 처음 등장한 모델은 뭐야?",

    # 의미 기반 검색
    "동부 소비자들이 버튼 방식에 익숙하지 않아서 지퍼를 적용한 501은 뭐야?",

    # 복합 조건 검색
    "여성용 최초의 5포켓 청바지이면서 Talon 지퍼를 사용하는 모델은 뭐야?",

    # 특정 명칭 기반 검색
    "501 Z로 불리는 지퍼 플라이 모델은 뭐야?"
]

for question in test_questions:
    result = rag_answer(question)

    print("\n" + "=" * 70)
    print("Q:", question)
    print("A:", result["answer"])


Q: 1944 501은 왜 아치형 스티치를 프린트로 만들었어?
A: 1944 501 ® JEANS는 제 2차 세계대전 중 자재 절약을 위한 미국 정부의 요청에 따라 금속, 천, 실을 제거하고 상품을 간소화하기로 결정했습니다. 이로 인해 아이코닉한 아치형 스티치를 제거하고 대신 모든 501 청바지 뒷주머니에 프린트로 상징적인 디자인을 적용하였습니다. 이는 자재 부족 상황에서 효율성을 높이기 위한 조치였습니다.

Q: 1954 501과 1950’S 701은 지퍼와 관련해서 각각 어떤 특징이 있어?
A: 1954 501 ® JEANS는 지퍼 플라이(지퍼가 있는 디자인)를 특징으로 하며, 501 ® Z라는 모델로도 알려져 있습니다. 반면, 1950’S 701 ® JEAN WOMEN’S는 Talon 지퍼를 사용한 지퍼 플라이 디자인입니다. 두 모델 모두 지퍼 플라이를 채택하고 있지만, 1954 501은 버튼 플라이 모델과 함께 판매되었고, 701은 여성용으로 제작된 최초의 5 포켓 청바지입니다.

Q: 1937과 1955 501의 데님 중량은 어떻게 달라?
A: 1937 501 ® JEANS의 데님 중량은 10oz이고, 1955 501 ® JEANS의 데님 중량은 12oz입니다. 따라서 1955 501 ® JEANS의 데님이 더 무겁습니다.

Q: 대문자 E 양면 레드탭을 사용하는 모델은 어떤 것들이야?
A: 조건에 해당하는 레드탭 모델은 다음과 같습니다:

- 1955 501 ® JEANS
- 1966 501 ® JEANS

Q: 무릎 폭이 가장 넓은 모델은 뭐야?
A: 무릎 폭 기준으로 조건에 해당하는 모델은 1955 501 ® JEANS이며, 값은 20 inch입니다.

Q: 1890 XX 501과 1955 501은 핏이 어떻게 달라?
A: 1890 XX 501은 하이 웨이스트의 와이드한 레그가 특징입니다. 반면, 1955 501은 허리를 조절하는 유일한 방법으로 벨트 루프를 남기고, 힙 부분에서 떨어지는 레그 라인에 여유가 있는 루즈한 실루엣을 가지고 있습니다

## 13. 결과 및 분석

Levi's Korea LVC Timeline 페이지를 직접 크롤링하여
JEAN/JEANS에 해당하는 총 10개의 바지 모델 정보를 수집하였다.

수집한 원문을 기반으로 모델별 FAQ를 생성한 뒤 직접 검수 및 보강하여
총 47개의 FAQ를 구성하였다. 또한 크롤링 원문을 23개의 Chunk로 분할하여,
FAQ와 원문 Chunk를 OpenAI Embedding으로 벡터화한 뒤 Chroma DB에 저장하였다.
최종적으로 Chroma DB에는 총 70개의 문서가 저장되었다.

초기에는 FAQ와 벡터 검색 결과만을 이용하여 답변을 생성했기 때문에,
전체 모델을 대상으로 하는 최대·최소 비교, 복수 조건 검색,
특정 조건에 해당하는 모든 모델을 찾는 질문에서 일부 정보가
누락되는 문제가 발생하였다.

이를 개선하기 위해 질문에 특정 연도가 포함된 경우에는 해당 모델의 공식 원문을,
전체 모델 비교가 필요한 경우에는 전체 공식 원문을 검색하도록
Retrieval 방식을 개선하였다.

또한 밑위, 무릎 폭, 밑단 폭, 원단 중량, 플라이, 레드탭과 같이
정확한 비교가 필요한 정보는 크롤링 원문을 직접 검수하여 구조화하였다.
수치 조건 검색, 최대·최소 계산, 특정 조건을 만족하는 모델 목록은
Python 코드가 구조화 데이터를 기반으로 직접 처리하도록 구성하였다.

반면 모델의 역사적 배경, 디자인 변화의 이유, 특징 비교와 같은
설명형 질문은 Chroma DB에서 검색한 Context를 기반으로 GPT가 답변하도록 하였다.
이를 통해 구조화 검색의 정확성과 자연어 기반 RAG의 설명 능력을 함께 활용하였다.

최종 테스트에서는 역사적 배경, 모델 비교, 수치 비교, 최대값 검색,
복합 조건 검색, 특징을 이용한 역검색 등 다양한 형태의 질문에 대해
수집한 데이터에 근거한 답변을 생성할 수 있었다.
또한 원문에 존재하지 않는 정보는 추측하지 않고
확인할 수 없다고 답변하도록 제한하였다.